In [0]:
from pyspark.sql import functions as F
from pyspark import pipelines as dp

In [0]:

CATALOG = spark.conf.get("catalog")
SILVER_SCHEMA = spark.conf.get("silver_schema")
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_view_events"
GOLD_SCHEMA = spark.conf.get("gold_schema")


DIM_SHOW = f"{CATALOG}.{GOLD_SCHEMA}.dim_show"
DIM_DATE = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"
DIM_USER = f"{CATALOG}.{GOLD_SCHEMA}.dim_user"
FACT_VIEWS = f"{CATALOG}.{GOLD_SCHEMA}.fact_views"

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_titles_by_type")
def gold_titles_by_type():
    return(
        spark.read.table(SILVER_TABLE)
        .groupBy("type")
        .agg(F.countDistinct("show_id").alias("title_count"))
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_titles_by_audience")
def gold_titles_by_audience():
    return(
        spark.read.table(SILVER_TABLE)
        .groupBy("audience_category")
        .agg(F.countDistinct("show_id").alias("title_count"))
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_titles_by_release_period")
def gold_titles_by_release_period():
    return(
        spark.read.table(SILVER_TABLE)
        .groupBy("release_period")
        .agg(F.countDistinct("show_id").alias("title_count"))
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_content")
def gold_content():
    df = spark.read.table(SILVER_TABLE)
    total_titles = df.count()
    return(
        df.groupBy(
            "type", "audience_category", "release_period"
        ).agg(
            F.countDistinct("show_id").alias("title_count")
        ).withColumn(
            "catalog_share", 
            F.round("title_count"/F.lit(total_titles)*100, 2)
        )
    ).orderBy(F.desc("title_count"))

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_views_by_show")
def gold_views_by_show():
    fact_df = spark.read.table(FACT_VIEWS)
    show_df = spark.read.table(DIM_SHOW)

    return(
        fact_df.join(show_df, on="show_key", how="left")
        .groupBy("show_key", "title", "type")
        .agg(F.sum("view_count").alias("total_views"),
             F.round(F.avg("rate"),2).alias("avg_rate"),
             F.countDistinct("user_key").alias("unique_viewers"))
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_views_by_date")
def gold_views_by_date():
    fact_df = spark.read.table(FACT_VIEWS)
    date_df = spark.read.table(DIM_DATE)

    return(
        fact_df.join(date_df, on="date_key", how="left")
        .groupBy("date_key", "view_date")
        .agg(F.sum("view_count").alias("total_views"),
             F.round(F.avg("rate"),2).alias("avg_rate"),
             F.countDistinct("user_key").alias("unique_viewers"))
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.gold_views_by_type")
def gold_views_by_type():
    fact_df = spark.read.table(FACT_VIEWS)
    show_df = spark.read.table(DIM_SHOW)

    return(
        fact_df.join(show_df, on="show_key", how="left")
        .groupBy("type")
        .agg(F.sum("view_count").alias("total_views"),
             F.round(F.avg("rate"),2).alias("avg_rate"),
             F.countDistinct("user_key").alias("unique_viewers"))
    )